# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mathu2112/FlyRank-ML-Internship-Repo/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

---

The week 4 rule remains the primary prioritisation signal because it outperformed the Decision Tree on the same held-out test set.The rule achieved an F1 score of 0.7768,compared with 0.6480 for the decision tree.

The decision tree was still useful for interpretation,but did not provide enough evidence to replace the simpler baseline.It relied mainly on gsc_avg_position,which accounted for 94.1% of its feature importance.


The action queue therefore uses the validated baseline and observed performance signals. These recommendations are decision-support only and require human review.
<br>

**Ranked actions:**

**HIGH — Investigate potential CTR opportunity.**

Prioritise content with at least 100 impressions, average search position between 1 and 20, and CTR below 0.005.

**Reason code: CTR_OPPORTUNITY**

**MEDIUM — Monitor content with weaker evidence.**

Content that does not meet the strongest CTR opportunity conditions should be monitored rather than immediately changed.

**Reason code: MONITOR**

The available dataset does not contain validated content-age, publication-date, last-update, content-type, or topic fields. Therefore, stale/fresh and archetype-specific actions are not generated by this queue.

These reason codes explain why an item entered the queue. They do not claim that taking the recommended action will cause CTR to improve.

---
**Archetype - action mapping**

Archetype → action mapping

The available dataset does not contain a validated content-type, topic, or archetype field. Therefore, this playbook does not create unsupported content archetypes.

Instead, actions are grouped by the observed CTR opportunity signal:

CTR opportunity → review search presentation and content relevance.
No CTR opportunity → monitor rather than take immediate action.

This keeps the recommendations tied to variables that were actually measured in the analysis.

**Decay/refresh insight**

The available dataset contains daily performance data but does not provide a validated content-age, publication-date, or last-update field. Therefore, this analysis cannot directly measure content decay or the effect of refreshing content.

Content refresh should remain a human decision rather than an automated recommendation. A future analysis could join validated content metadata containing publication or last-update dates and test whether CTR opportunity rates differ across content-age groups.

No causal claim about freshness or refreshing content is made in this playbook.

In [6]:
# ML-10 Section 1 — Load data and create ranked action queue

!pip install -q datasets pandas pyarrow huggingface_hub

from google.colab import userdata
from datasets import load_dataset
import numpy as np
import pandas as pd

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Load the same dataset used in ML-08
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

# Select train split
data = dataset["train"]

print("Dataset loaded:", len(data), "rows")

# Use the same 1,000,000-row sample as ML-08
sample = data.select(range(1_000_000))
action_df = sample.to_pandas()

print("Sample shape:", action_df.shape)

# Calculate CTR
action_df["ctr"] = np.where(
    action_df["gsc_impressions"] > 0,
    action_df["gsc_clicks"] / action_df["gsc_impressions"],
    np.nan
)

# Keep the same eligibility conditions used in ML-08
action_df = action_df[
    (action_df["gsc_data_available"] == True) &
    (action_df["gsc_impressions"] >= 100) &
    (action_df["gsc_impressions"] > 0)
].copy()

print("Eligible rows:", len(action_df))

# Week-4 validated CTR opportunity rule
action_df["ctr_opportunity"] = (
    (action_df["gsc_avg_position"] > 0) &
    (action_df["gsc_avg_position"] <= 20) &
    (action_df["gsc_impressions"] >= 100) &
    (action_df["ctr"] < 0.005)
)

# Assign reason codes
action_df["reason_code"] = "MONITOR"
action_df["action"] = "Monitor; no immediate action"
action_df["priority_rank"] = 2

# High-priority opportunities
action_df.loc[action_df["ctr_opportunity"], "reason_code"] = "CTR_OPPORTUNITY"

action_df.loc[action_df["ctr_opportunity"], "action"] = (
    "Review search presentation and content relevance"
)

action_df.loc[action_df["ctr_opportunity"], "priority_rank"] = 1

# Rank stronger opportunities first
action_df["ctr_gap"] = 0.005 - action_df["ctr"]

queue = action_df.sort_values(
    ["priority_rank", "ctr_gap"],
    ascending=[True, False]
).copy()

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

print("\nTop 20 action queue:")

display(
    queue[
        [
            "reason_code",
            "action",
            "priority_rank",
            "ctr",
            "gsc_impressions",
            "gsc_avg_position"
        ]
    ].head(20)
)

# ML-10 — Measure the observed decay/refresh pattern

# Find the best available age grouping
if "decay_band" in action_df.columns:
    decay_summary = (
        action_df
        .groupby("decay_band", dropna=False)
        .agg(
            observations=("ctr", "size"),
            median_ctr=("ctr", "median"),
            ctr_opportunity_rate=("ctr_opportunity", "mean")
        )
        .reset_index()
    )

    decay_summary["ctr_opportunity_rate"] *= 100

    print("Observed performance by decay band:")
    display(decay_summary)

elif "content_age_days" in action_df.columns:
    action_df["age_band"] = pd.cut(
        action_df["content_age_days"],
        bins=[-1, 30, 90, 180, np.inf],
        labels=["0-30d", "31-90d", "91-180d", "180d+"]
    )

    decay_summary = (
        action_df
        .groupby("age_band", observed=False)
        .agg(
            observations=("ctr", "size"),
            median_ctr=("ctr", "median"),
            ctr_opportunity_rate=("ctr_opportunity", "mean")
        )
        .reset_index()
    )

    decay_summary["ctr_opportunity_rate"] *= 100

    print("Observed performance by content age:")
    display(decay_summary)

else:
    print("No decay/age field found.")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset loaded: 78835655 rows
Sample shape: (1000000, 30)
Eligible rows: 67650

Reason-code counts:
reason_code
MONITOR            35935
CTR_OPPORTUNITY    31715
Name: count, dtype: int64

Top 20 action queue:


,reason_code,action,priority_rank,ctr,gsc_impressions,gsc_avg_position
45,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,118,3.440678
1121,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,102,4.303922
1588,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,115,15.486957
2292,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,118,5.830508
2699,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,118,2.661017
3096,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,105,15.371429
3825,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,424,1.099057
3935,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,118,8.991525
3957,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,108,5.777778
4400,CTR_OPPORTUNITY,Review search presentation and content relevance,1,0.0,142,2.845070


No decay/age field found.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

---

The playbook is intended for content teams or analysts who need a shortlist of content to investigate for possible CTR improvement.
It combines the validated Week-4 rule with observed content and freshness signals to prioritise human review.

The Week-4 rule is preferred over the Decision Tree for prioritisation because it performed better on the held-out test set. The Decision Tree should therefore not be presented as a superior replacement.

The playbook is not a production decision system. It does not prove that changing content will increase CTR, and it should not automatically publish, remove, rewrite, or refresh content.

The recommendations are most appropriate for observations that are similar to the data used in this analysis. Changes in search behaviour, content mix, measurement definitions, or data quality may reduce their usefulness.

In [7]:
# This cell is for CODE (numbers, a query, a check).


intended_use_checks = pd.DataFrame({
    "rule": [
        "Decision support only",
        "Human review required",
        "No causal claim",
        "Not an automatic publishing system",
        "Use only for data reasonably similar to the analysis data"
    ],
    "status": [
        "PASS",
        "REQUIRED",
        "REQUIRED",
        "REQUIRED",
        "REQUIRED"
    ]
})

display(intended_use_checks)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,rule,status
0,Decision support only,PASS
1,Human review required,REQUIRED
2,No causal claim,REQUIRED
3,Not an automatic publishing system,REQUIRED
4,Use only for data reasonably similar to the an...,REQUIRED


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

---

## 3. Human review + the no-go list

Every high-priority recommendation should be reviewed by a person before action.

The reviewer should check:

1. Whether the impression and CTR measurements are valid.
2. Whether the content is still relevant to the current search intent.
3. Whether the content has already been recently changed.
4. Whether there is another reason for the observed CTR pattern.
5. Whether the recommended action is practical and worthwhile.
6. Whether the observation is unusual compared with the data used for the analysis.

### No-go list

The following should never be automated using this playbook:

* Automatically publishing content changes.
* Automatically deleting or de-prioritising content.
* Automatically rewriting titles or descriptions.
* Treating model feature importance as causal evidence.
* Treating a low CTR prediction as proof that content is poor.
* Automatically refreshing every stale page.
* Making high-impact decisions without human review.

---

** Cost/value thinking **

The highest-ranked item is not automatically the highest-value item.A practical action should consider both the strength of the observed signal and the cost of making the change.

Low-cost reviews,such as checking a title or search-result presentation,can be considered before expensive content rewrites.

A useful prioritisation order is:

Strong observed opportunity + low action cost → review first.

Strong observed opportunity + high action cost → investigate carefully before committing resources.

Weak observed signal + high action cost → monitor rather than act immediately.

This is a prioritisation framework rather than a calculated financial return. No monetary value is claimed because the analysis does not measure the business value of individual content changes.


In [9]:
# This cell is for CODE (numbers, a query, a check).



review_checklist = pd.DataFrame({
    "review_check": [
        "Data quality verified",
        "Content relevance checked",
        "Recent changes checked",
        "Potential alternative explanation considered",
        "Cost/value considered",
        "Human approval obtained"
    ],
    "required_before_action": [True] * 6
})

display(review_checklist)

# ML-10 — Simple cost/value priority framework

cost_value = pd.DataFrame({
    "signal_strength": ["Strong", "Strong", "Weak"],
    "action_cost": ["Low", "High", "High"],
    "recommended_priority": ["High", "Medium", "Low"],
    "reason": [
        "Easy to investigate with strong observed signal",
        "Potentially useful but requires more review",
        "Evidence is weak relative to action cost"
    ]
})

display(cost_value)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,review_check,required_before_action
0,Data quality verified,True
1,Content relevance checked,True
2,Recent changes checked,True
3,Potential alternative explanation considered,True
4,Cost/value considered,True
5,Human approval obtained,True


,signal_strength,action_cost,recommended_priority,reason
0,Strong,Low,High,Easy to investigate with strong observed signal
1,Strong,High,Medium,Potentially useful but requires more review
2,Weak,High,Low,Evidence is weak relative to action cost


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

---

The playbook should be reviewed if the observed data changes enough that the current recommendations may no longer represent the current content environment.

Useful triggers are:

F1 or another selected evaluation metric falls materially below the Week-5 result.
The proportion of CTR-opportunity cases changes substantially.
The distribution of search positions changes.
The distribution of important content or decay groups changes.
New content types appear that were not represented in the analysis.
The definition or collection of CTR changes.
The observed relationship between content age and CTR opportunity changes.
Human reviewers repeatedly reject the recommendations.

A retrain or re-analysis should be considered after a meaningful data or measurement change, rather than on an arbitrary schedule.

In [10]:
# This cell is for CODE (numbers, a query, a check).

# ML-10 Section 4 — Monitoring thresholds and checks

week5_f1 = 0.6480
baseline_f1 = 0.7768

monitoring = {
    "week5_decision_tree_f1": week5_f1,
    "week4_baseline_f1": baseline_f1,
    "human_review_required": True,
    "retrain_trigger": "Material performance/data distribution/measurement change"
}

for key, value in monitoring.items():
    print(f"{key}: {value}")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


week5_decision_tree_f1: 0.648
week4_baseline_f1: 0.7768
human_review_required: True
retrain_trigger: Material performance/data distribution/measurement change


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*
---

he notebook exports the ranked action queue and supporting summaries to work/outputs/.

The queue is regenerated by the notebook rather than committed as raw data. Summary metrics and reusable figures can be committed separately as required by the repository instructions.

The exported results are intended to support the recommendations section of the research paper. They describe observed model and data patterns and should not be interpreted as evidence of causal impact.

In [11]:
# This cell is for CODE (numbers, a query, a check).

# ML-10 Section 5 — Export the queue and supporting summaries

import os
import json

os.makedirs("work/outputs", exist_ok=True)

# Select useful queue columns.
queue_columns = [
    "reason_code",
    "action",
    "priority_rank",
    "ctr",
    "gsc_impressions",
    "gsc_avg_position"
]

# Add optional fields if they exist.
for col in [
    "content_type",
    "topic_cluster",
    "content_age_days",
    "days_since_update",
    "decay_band",
    "refresh_flag",
    "is_fresh_30d",
    "is_stale_90d",
    "is_stale_180d"
]:
    if col in queue.columns:
        queue_columns.append(col)

paper_queue = queue[queue_columns].copy()

# Export ranked queue.
paper_queue.to_csv(
    "work/outputs/ml10_ranked_action_queue.csv",
    index=False
)

# Export decay summary if it was calculated.
if "decay_summary" in globals():
    decay_summary.to_csv(
        "work/outputs/ml10_decay_summary.csv",
        index=False
    )

# Export validated model comparison as a small metrics receipt.
metrics = {
    "week4_baseline_f1": 0.7768,
    "decision_tree_f1": 0.6480,
    "decision_tree_top_feature": "gsc_avg_position",
    "decision_tree_top_feature_importance": 0.941,
    "false_negatives": 2516,
    "false_negative_mean_position": 6.35
}

with open(
    "work/outputs/ml10_metrics_receipt.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print("Exports created:")
print("- work/outputs/ml10_ranked_action_queue.csv")
print("- work/outputs/ml10_metrics_receipt.json")

if "decay_summary" in globals():
    print("- work/outputs/ml10_decay_summary.csv")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Exports created:
- work/outputs/ml10_ranked_action_queue.csv
- work/outputs/ml10_metrics_receipt.json


## Self-check

Before you submit, confirm each line honestly:

- [ Yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes ] No client names, URLs, or private queries anywhere
- [ Yes ] My claims use careful words: observed, measured, directional, decision-support
- [ Yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.